# Direction Policy Kaggle Pipeline

This notebook prepares raw M5 forex CSV data, generates direction labels, and trains/replays one or more of the five neural architectures.

Edit the paths in the first code cell to match your Kaggle datasets.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

# === EDIT THESE ===
PROJECT_ZIP = '/kaggle/input/YOUR_PROJECT_DATASET/direction_policy_kaggle_ready.zip'
RAW_DATASET_DIR = '/kaggle/input/YOUR_RAW_FOREX_DATASET'
SYMBOLS = ['EURUSD']
TRAIN_START = '2024-01-01'
TRAIN_END = '2025-03-01'
REPLAY_START = '2025-03-01'
REPLAY_END = '2025-06-01'
EPOCHS = 50
BATCH_SIZE = 512
DEVICE = 'cuda'  # use 'cpu' if no GPU accelerator is enabled

WORKDIR = Path('/kaggle/working/project')
WORKDIR.parent.mkdir(parents=True, exist_ok=True)

if not WORKDIR.exists():
    if Path(PROJECT_ZIP).exists():
        with zipfile.ZipFile(PROJECT_ZIP, 'r') as zf:
            zf.extractall(WORKDIR)
    else:
        raise FileNotFoundError(f'Edit PROJECT_ZIP. Not found: {PROJECT_ZIP}')

os.chdir(WORKDIR)
print('Project directory:', Path.cwd())
print('Raw dataset directory:', RAW_DATASET_DIR)


In [ ]:
# Install Kaggle-safe requirements. MetaTrader5 is intentionally not installed.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements_kaggle.txt'], check=True)


In [ ]:
# Optional smoke test: set these for a fast small run. For full training, leave as None.
RAW_MAX_ROWS_PER_SYMBOL = None      # e.g. 120000
TRAIN_MAX_ROWS = None               # e.g. 30000
MODE = 'train-all'                  # 'prepare-only', 'train-one', 'train-all', or 'grid'
CONFIGS = [
    'config/direction_settings_residual_mlp.yaml',
    'config/direction_settings_tcn.yaml',
    'config/direction_settings_inception_time.yaml',
    'config/direction_settings_small_transformer.yaml',
    'config/direction_settings_mixture_of_experts.yaml',
]

cmd = [
    sys.executable, 'kaggle_run_pipeline.py',
    '--raw-input-dir', RAW_DATASET_DIR,
    '--symbols', *SYMBOLS,
    '--timeframe', 'M5',
    '--train-start', TRAIN_START,
    '--train-end', TRAIN_END,
    '--replay-start', REPLAY_START,
    '--replay-end', REPLAY_END,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--device', DEVICE,
    '--mode', MODE,
    '--configs', *CONFIGS,
]
if RAW_MAX_ROWS_PER_SYMBOL is not None:
    cmd += ['--raw-max-rows-per-symbol', str(RAW_MAX_ROWS_PER_SYMBOL)]
if TRAIN_MAX_ROWS is not None:
    cmd += ['--train-max-rows', str(TRAIN_MAX_ROWS)]

print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Package outputs for download from the Kaggle notebook output panel.
output_zip = Path('/kaggle/working/direction_policy_outputs.zip')
items = ['models', 'logs', 'data/direction', 'config/generated_spread_risk.yaml']
with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for item in items:
        p = Path(item)
        if not p.exists():
            continue
        if p.is_file():
            zf.write(p, p.as_posix())
        else:
            for f in p.rglob('*'):
                if f.is_file():
                    zf.write(f, f.as_posix())
print('Wrote', output_zip)
